# 5. Creating the Stuff Documents Chain

This notebook turns retrieved chunks into a single prompt payload that the model can answer from.

What to look for:
- how the prompt expects context
- how retrieved documents are combined into one input
- why this is a simple but effective first answer-generation strategy

In [1]:
from pathlib import Path
from hashlib import sha256
from datetime import datetime, timezone
from uuid import uuid4
import os

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

C:\Users\admin\AppData\Local\Temp\ipykernel_13876\2148886914.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [2]:
load_dotenv()
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [3]:
DATA_DIR = Path("data")

In [4]:
persist_directory = DATA_DIR / "bookstore"

In [5]:
pdf_files_paths = DATA_DIR.glob("*.pdf")

In [6]:
list(pdf_files_paths)

[WindowsPath('data/Atomic habits ( PDFDrive ).pdf'),
 WindowsPath('data/attention.pdf'),
 WindowsPath('data/BhagavadGita.pdf'),
 WindowsPath('data/the-5-am-club.pdf')]

In [7]:
def normalize_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text.lower()).strip("_")


def file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    hasher = sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            hasher.update(block)
    return hasher.hexdigest()

In [8]:
all_documents = []
source_ids = set() 

for pdf_file_path in DATA_DIR.glob("*.pdf"):
    print(f"{pdf_file_path.name}: {file_sha256(pdf_file_path)}")
    loader = PyMuPDFLoader(str(pdf_file_path))
    documents = loader.load()

    source_checksum = file_sha256(pdf_file_path)
    source_id = normalize_name(pdf_file_path.stem)
    source_ids.add(source_id)

    for index, doc in enumerate(documents):
        doc.metadata["source"] = pdf_file_path.name
        doc.metadata["source_id"] = source_id
        doc.metadata["source_checksum"] = source_checksum
        doc.metadata["page_number"] = index + 1
        doc.metadata["chunk_id"] = str(uuid4())
        doc.metadata["chunk_index"] = index
        doc.metadata["chunk_size"] = len(doc.page_content)
        doc.metadata["created_at"] = datetime.now(timezone.utc).isoformat()

    all_documents.extend(documents)

documents = all_documents
print(len(documents), "documents loaded from all PDFs.")
sorted(set(source_ids))

Atomic habits ( PDFDrive ).pdf: a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878


attention.pdf: bdfaa68d8984f0dc02beaca527b76f207d99b666d31d1da728ee0728182df697
BhagavadGita.pdf: ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583
the-5-am-club.pdf: 089bb3fc5b41e7de713444b93214434e1d887c47dff66e8e4ae075aeed89440b
1475 documents loaded from all PDFs.


['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

In [10]:
chunks = splitter.split_documents(documents)
chunks[0].metadata

{'producer': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]',
 'creationdate': '2020-04-30T18:46:22+00:00',
 'source': 'Atomic habits ( PDFDrive ).pdf',
 'file_path': 'data\\Atomic habits ( PDFDrive ).pdf',
 'total_pages': 256,
 'format': 'PDF 1.4',
 'title': 'Atomic habits \\( PDFDrive.com \\).pdf',
 'author': 'James Clear',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': "D:20200430184622+00'00'",
 'page': 2,
 'source_id': 'atomic_habits___pdfdrive',
 'source_checksum': 'a8cacf49ea808daacbef95466b3bcbd3db371d63fdeabcb8d26cf15504062878',
 'page_number': 3,
 'chunk_id': 'fc123ac6-a1b8-4912-8aef-03ba4312243d',
 'chunk_index': 2,
 'chunk_size': 531,
 'created_at': '2026-07-28T19:56:34.606816+00:00'}

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [12]:
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))
collection_names = [normalize_name(p.stem) for p in pdf_paths]
collection_names

['atomic_habits___pdfdrive', 'attention', 'bhagavadgita', 'the_5_am_club']

In [13]:
collection_store = {}

for collection in collection_names:
    collection_path = persist_directory / collection

    if not collection_path.exists():
        collection_path.mkdir(parents=True, exist_ok=True)

    temp_store = Chroma(
        persist_directory=str(collection_path),
        embedding_function=embeddings,
        collection_name=collection,
    )

    try:
        temp_store._client.delete_collection(name=collection)
    except Exception:
        pass

    collection_store[collection] = Chroma(
        persist_directory=str(collection_path),
        embedding_function=embeddings,
        collection_name=collection,
    )

collection_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x1c326105f70>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1c326347b60>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1c3261ef5c0>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x1c3263d3110>}

In [14]:
missing_source_ids = set()
inserted_count = 0
batch_size = 100

chunks_by_source = {}
for chunk in chunks:
    source_id = chunk.metadata["source_id"]
    if source_id not in chunks_by_source:
        chunks_by_source[source_id] = []
    chunks_by_source[source_id].append(chunk)

for source_id, source_chunks in chunks_by_source.items():
    vector_store = collection_store.get(source_id)
    if not vector_store:
        missing_source_ids.add(source_id)
        continue

    for start in range(0, len(source_chunks), batch_size):
        end = start + batch_size
        vector_store.add_documents(source_chunks[start:end])
        inserted_count += len(source_chunks[start:end])

print("Inserted chunks:", inserted_count)
print("Missing source_id keys:", sorted(missing_source_ids))

Inserted chunks: 3583
Missing source_id keys: []


# Creating retriever for the RAG system

In [15]:
query = ['What does krishna says about the Mahabharat battle']

In [16]:
collection_store

{'atomic_habits___pdfdrive': <langchain_chroma.vectorstores.Chroma at 0x1c326105f70>,
 'attention': <langchain_chroma.vectorstores.Chroma at 0x1c326347b60>,
 'bhagavadgita': <langchain_chroma.vectorstores.Chroma at 0x1c3261ef5c0>,
 'the_5_am_club': <langchain_chroma.vectorstores.Chroma at 0x1c3263d3110>}

In [17]:
bg_vectorstore = collection_store.get('bhagavadgita')
bg_vectorstore

In [50]:
bg_retriver = bg_vectorstore.as_retriever(search_kwargs={"k": 5})

In [19]:
bg_retriver.invoke(input=query[0])

[Document(id='7f669047-a734-47bd-90c1-0acc50817788', metadata={'format': 'PDF 1.3', 'creationdate': "D:20120508142201Z00'00'", 'page_number': 51, 'author': 'me', 'chunk_id': '4c9aebd3-d491-4219-9407-a739b18af999', 'file_path': 'data\\BhagavadGita.pdf', 'created_at': '2026-07-28T19:56:37.418572+00:00', 'source': 'BhagavadGita.pdf', 'source_id': 'bhagavadgita', 'keywords': '', 'total_pages': 952, 'title': 'Bhagavad-gita As It Is with pics!', 'creationDate': "D:20120508142201Z00'00'", 'modDate': "D:20120508142201Z00'00'", 'producer': 'Mac OS X 10.7.3 Quartz PDFContext', 'trapped': '', 'chunk_index': 50, 'moddate': "D:20120508142201Z00'00'", 'chunk_size': 1333, 'subject': '', 'page': 50, 'creator': 'Pages', 'source_checksum': 'ff112b0b056d303b792f6f2e68cbd73a89adf612fa9113f932446cdea7741583'}, page_content='position as a lion. Indirectly, by the symbolism of the conchshell, he informed \nhis depressed grandson Duryodhana that he had no chance of victory in the \nbattle, because the Supreme

# Building a Part 2 - RAG Pipeline

## Creating Stuff Documents Chain

In [20]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [24]:
from langchain_core.prompts import ChatPromptTemplate

In [23]:
system_prompt = '''

You are an assistant that answers questions based on the context provided. You are given a question and a set of documents that may contain the answer.
Your task is to provide a concise and accurate answer to the question using the information from the documents. 
If the answer is not present in the documents, respond with "I don't know."

context:{context}

'''

In [37]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [33]:
from langchain.chat_models.base import init_chat_model

In [ ]:
llm = init_chat_model(
    model='gpt-5.4-nano',
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0.0,
    max_tokens=512,
) 

In [35]:
response = llm.invoke("Hi")
response.content


'Hi! How can I help you today?'

In [ ]:
combine_docs_chain = create_stuff_documents_chain(
    llm, prompt
)
combine_docs_chain